<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/01_fatura_dataset_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
import os
import json
from torch.utils.data import Dataset
import random

In [2]:
!wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O /content/FATURA2.zip

--2026-10-10 18:09:18--  https://zenodo.org/records/10371464/files/FATURA2.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 690717658 (659M) [application/octet-stream]
Saving to: ‘/content/FATURA2.zip’

/content/FATURA2.zi 100%[===================>] 658.72M  1.48MB/s    in 7m 40s  

2026-10-10 18:17:00 (1.43 MB/s) - ‘/content/FATURA2.zip’ saved [690717658/690717658]



In [3]:
!unzip -q /content/FATURA2.zip -d /content/FATURA

In [4]:
print(os.listdir("/content/FATURA"))

['invoices_dataset_final']


In [5]:
base_path = "/content/FATURA/invoices_dataset_final"
print(os.listdir(base_path))

['Annotations', 'images', 'Strat2_Split.txt', 'strat1_test.csv', 'strat1_train.csv', 'strat1_dev.csv', 'colored_images']


In [6]:
train  = pd.read_csv(f'{base_path}/strat1_train.csv')
val = pd.read_csv(f'{base_path}/strat1_dev.csv')
test = pd.read_csv(f"{base_path}/strat1_test.csv")


In [7]:
print("Shape of Train Data:",train.shape)
print("Shape of Val Data:",val.shape)
print("Shape of Test Data:",test.shape)
print()
print()
train.head()

Shape of Train Data: (7500, 2)
Shape of Val Data: (1250, 2)
Shape of Test Data: (1250, 2)




,img_path,annot_path
0,Template1_Instance0.jpg,Template1_Instance0.json
1,Template1_Instance1.jpg,Template1_Instance1.json
2,Template1_Instance2.jpg,Template1_Instance2.json
3,Template1_Instance3.jpg,Template1_Instance3.json
4,Template1_Instance4.jpg,Template1_Instance4.json


In [8]:
os.listdir('/content/FATURA/invoices_dataset_final/Annotations')

['.virtual_documents',
 'Original_Format',
 'layoutlm_HF_format',
 'COCO_compatible_format']

In [9]:
train.head()

,img_path,annot_path
0,Template1_Instance0.jpg,Template1_Instance0.json
1,Template1_Instance1.jpg,Template1_Instance1.json
2,Template1_Instance2.jpg,Template1_Instance2.json
3,Template1_Instance3.jpg,Template1_Instance3.json
4,Template1_Instance4.jpg,Template1_Instance4.json


In [10]:
def template_ids(data):
    for temp_data in data.columns:
        values = data[temp_data].astype(str)
        template = values.str.extract(r"Template(\d+)")[0]

        print(temp_data)
        print(template.dropna().head())
template_ids(train)

img_path
0    1
1    1
2    1
3    1
4    1
Name: 0, dtype: object
annot_path
0    1
1    1
2    1
3    1
4    1
Name: 0, dtype: object


In [11]:
img = Image.open(
    f'{base_path}/images/Template1_Instance0.jpg'
)
print(img.size)
print(img.mode)

(595, 841)
RGB


In [12]:
class FATURADataset(Dataset):
    def __init__(self, dataframe, base_path):
        self.dataframe = dataframe
        self.base_path = base_path

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        sample = self.dataframe.iloc[index]

        return sample


# Making Dataset as object

train_dataset = FATURADataset(train, base_path)
val_dataset = FATURADataset(val, base_path)
test_dataset = FATURADataset(test, base_path)

# Check lengths
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

# Get one sample
sample = train_dataset[1]

print("\nSample:")
print(sample)

Train: 7500
Validation: 1250
Test: 1250

Sample:
img_path       Template1_Instance1.jpg
annot_path    Template1_Instance1.json
Name: 1, dtype: object


In [13]:
train['template_id']=train['img_path'].str.extract(r'Template(\d+)')[0]
val['template_id'] = val['img_path'].str.extract(r'Template(\d+)')[0]
test['template_id'] = test['img_path'].str.extract(r'Template(\d+)')[0]


overlap_id = set(train['template_id']) & set(test['template_id'])
print(overlap_id)

{'22', '29', '4', '34', '50', '10', '5', '27', '36', '47', '8', '3', '14', '48', '39', '32', '26', '38', '1', '15', '12', '24', '11', '6', '18', '19', '9', '31', '33', '25', '23', '49', '7', '16', '20', '30', '41', '17', '37', '40', '46', '42', '35', '43', '21', '45', '2', '44', '13', '28'}


In [14]:
sorted_overlap = sorted([int(x) for x in overlap_id])
print("Sorted Overlap:", sorted_overlap)

Sorted Overlap: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


In [15]:
df = pd.concat([train,val,test],ignore_index=True)
print(df.shape)

(10000, 3)


In [16]:
unique_templates = df['template_id'].drop_duplicates().tolist()
print(unique_templates)

print()
print(len(unique_templates))

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50']

50


In [17]:
random.seed(42)
random.shuffle(unique_templates)
print(unique_templates)

train_temps = unique_templates[:32]
val_temps = unique_templates[32:40]
test_temps = unique_templates[40:]

new_train = df[df['template_id'].isin(train_temps)].copy()
new_val = df[df['template_id'].isin(val_temps)].copy()
new_test = df[df['template_id'].isin(test_temps)].copy()



print("New Train Rows:",new_train.shape[0])
print("New Val Rows:",new_val.shape[0])
print("New Test Rows:",new_test.shape[0])

overlap_id = set(new_train['template_id']) & set(new_test['template_id'])

print("Overlap ID:",overlap_id)

['26', '24', '20', '12', '5', '46', '27', '10', '30', '17', '32', '22', '13', '4', '40', '39', '11', '25', '36', '1', '44', '19', '34', '49', '42', '31', '29', '21', '23', '43', '47', '37', '33', '45', '14', '50', '48', '3', '28', '38', '6', '35', '7', '9', '15', '16', '18', '2', '8', '41']
New Train Rows: 6400
New Val Rows: 1600
New Test Rows: 2000
Overlap ID: set()


In [18]:
new_train.to_csv('custom_train_split.csv', index=False)
new_val.to_csv('custom_val_split.csv', index=False)
new_test.to_csv('custom_test_split.csv', index=False)

In [19]:
!pip install -q huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [20]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "Satyamkarn100/FinDoc-AI"

files_to_push = ['custom_train_split.csv', 'custom_val_split.csv', 'custom_test_split.csv']

for file in files_to_push:
    api.upload_file(
        path_or_fileobj=file,
        path_in_repo=f"dataset_splits/{file}",
        repo_id=repo_id,
        repo_type="dataset"
    )
